# Content-Based Movie Recommender
A simple recommender built with pandas on the MovieLens `movies.csv` file.

**Idea:** build a *taste profile* for a user from the genres of the movies they rated (ratings are centered around a neutral midpoint, so low ratings count negatively), then rank all movies by the cosine similarity between their genres and that profile. Movies the user already rated are removed.

## 1. Load the data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../movies.csv')

In [3]:
df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34208 entries, 0 to 34207
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  34208 non-null  int64
 1   title    34208 non-null  str  
 2   genres   34208 non-null  str  
dtypes: int64(1), str(2)
memory usage: 801.9 KB


In [5]:
df.shape

(34208, 3)

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.columns

Index(['movieId', 'title', 'genres'], dtype='str')

## 2. Clean the data
Split the year out of the title, e.g. `Toy Story (1995)` -> `Toy Story` and `1995`.

In [8]:
df["year"] = df["title"].str.extract(r"\((\d{4})\)").astype("Int64")
df["title"]= df["title"].str.replace(r"\((\d{4})\)" , "" ,regex=True).str.strip()

In [9]:
df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


## 3. One-hot encode the genres
Each genre becomes a 0/1 column. Movies with no genre are removed.

In [10]:
dummies = df["genres"].str.get_dummies(sep="|")
df = pd.concat([df , dummies] , axis=1)

In [11]:
print(df.shape)
df= df[df["(no genres listed)"] == 0]
df.shape

(34208, 24)


(33063, 24)

In [12]:
df = df.drop(["genres" ,"(no genres listed)" ] , axis=1)
df.head()

,movieId,title,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story,1995,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji,1995,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men,1995,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale,1995,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II,1995,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


Remove movies whose year is missing.

In [13]:
df = df.dropna()
df.isna().sum()

movieId        0
title          0
year           0
Action         0
Adventure      0
Animation      0
Children       0
Comedy         0
Crime          0
Documentary    0
Drama          0
Fantasy        0
Film-Noir      0
Horror         0
IMAX           0
Musical        0
Mystery        0
Romance        0
Sci-Fi         0
Thriller       0
War            0
Western        0
dtype: int64

## 4. Example user ratings
A small made-up user with ratings on a 0-10 scale.

In [14]:
user = pd.DataFrame({
    "userId": [1, 1, 1, 1, 1, 1],
    "title": [
        "Apollo 13 (1995)",
        "Heat (1995)",
        "Creed (2015)",
        "Interstellar (2014)",
        "Se7en (1995)",
        "Mad Max: Fury Road (2015)"
    ],
    "rating": [8.5, 3.0, 8, 10.0, 5.0, 7.5]
})
user


,userId,title,rating
0,1,Apollo 13 (1995),8.5
1,1,Heat (1995),3.0
2,1,Creed (2015),8.0
3,1,Interstellar (2014),10.0
4,1,Se7en (1995),5.0
5,1,Mad Max: Fury Road (2015),7.5


In [15]:
user = user.drop([ "userId"], axis=1)
user["year"] = user.title.str.extract(r"\((\d{4})\)").astype("Int64")
user["title"] = user.title.str.replace(r"\((\d{4})\)" , "" ,regex=True).str.strip()
user


,title,rating,year
0,Apollo 13,8.5,1995
1,Heat,3.0,1995
2,Creed,8.0,2015
3,Interstellar,10.0,2014
4,Se7en,5.0,1995
5,Mad Max: Fury Road,7.5,2015


In [16]:
df[df["title"].str.contains("seven|se7en" , case=False , regex=True) & (df["year"] == 1995)]

,movieId,title,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
46,47,Seven (a.k.a. Se7en),1995,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
24588,115607,"Seventh Brother, The (A hetedik testvér)",1995,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Se7en is stored as `Seven (a.k.a. Se7en)` in the dataset, so a merge on the title would silently drop it. We add a row with the exact dataset title.

In [17]:
row = pd.DataFrame(df[df["movieId"]==47][["title","year"]]).copy()
row["rating"] = 5.0
user = pd.concat([user , row] , ignore_index=True)
user

,title,rating,year
0,Apollo 13,8.5,1995
1,Heat,3.0,1995
2,Creed,8.0,2015
3,Interstellar,10.0,2014
4,Se7en,5.0,1995
5,Mad Max: Fury Road,7.5,2015
6,Seven (a.k.a. Se7en),5.0,1995


## 5. Center the ratings
Subtract a neutral midpoint: above it a rating is positive, below it negative, and the further away, the stronger the effect.

In [18]:
midpoint = 5   # neutral rating on a 0-10 scale (use 2.75 for MovieLens' 0.5-5 scale)

# Run this cell only once: it changes "rating" in place. Use Restart & Run All to redo it.
user["rating"] = user["rating"] - midpoint

Remove the old `Se7en` row, which never matches the dataset title.

In [19]:
user = user[user["title"] != "Se7en"]

## 6. Match the user's movies with the dataset

In [20]:
user_movies = df.merge(user , on=["title" , "year"] ,how='inner')
user_movies

,movieId,title,year,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,rating
0,6,Heat,1995,1,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,-2.0
1,47,Seven (a.k.a. Se7en),1995,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.0
2,150,Apollo 13,1995,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,3.5
3,109487,Interstellar,2014,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,5.0
4,122882,Mad Max: Fury Road,2015,1,1,0,0,0,0,0,...,0,0,0,0,0,1,1,0,0,2.5
5,146656,Creed,2015,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3.0


In [21]:
user_profile = user_movies [[ "title" , "year" , "rating" , "movieId"]]
user_profile = user_profile.set_index("movieId")
user_profile

,title,year,rating
movieId,,,
6,Heat,1995,-2.0
47,Seven (a.k.a. Se7en),1995,0.0
150,Apollo 13,1995,3.5
109487,Interstellar,2014,5.0
122882,Mad Max: Fury Road,2015,2.5
146656,Creed,2015,3.0


In [22]:
user_movies= user_movies.drop(["movieId" , "title" ,"year" ], axis=1)

In [23]:
user_rate = user_movies["rating"].tolist()
user_movies = user_movies.drop("rating" , axis=1)

## 7. Build the user's taste profile
Multiply each movie's genres by its centered rating, then sum per genre.

In [24]:
genre_rating = user_movies.mul(user_rate , axis=0)
genre_rating_sum = genre_rating.sum()
genre_rating_sum

Action         0.5
Adventure      6.0
Animation      0.0
Children       0.0
Comedy         0.0
Crime         -2.0
Documentary    0.0
Drama          6.5
Fantasy        0.0
Film-Noir      0.0
Horror         0.0
IMAX           8.5
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         7.5
Thriller       0.5
War            0.0
Western        0.0
dtype: float64

## 8. Prepare the genre matrix of all movies

In [25]:
df_genres = df.drop([ "title" , "year"] , axis=1)
print(df_genres["movieId"].is_unique)

df_genres = df_genres.set_index("movieId")

True


## 9. Score every movie with cosine similarity

In [26]:
profile = genre_rating_sum                  # taste vector: one number per genre

dot = df_genres.dot(profile)                # per movie: sum of (genre flag * taste value)
n_genres = df_genres.sum(axis=1)            # number of genres of each movie
movie_norm = n_genres ** 0.5                # length of each movie's 0/1 genre vector
profile_norm = (profile ** 2).sum() ** 0.5  # length of the taste vector

# Cosine similarity = dot / (|movie| * |profile|)
recomend = (dot / (movie_norm * profile_norm)).sort_values(ascending=False)
recomend.head(10)

movieId
110730    0.893765
97752     0.893765
110127    0.834181
97938     0.834181
150       0.834181
1019      0.794458
329       0.794458
142863    0.794458
3471      0.794458
148080    0.794458
dtype: float64

## 10. Remove movies the user already rated

In [27]:
watched_ids = df.merge(user , on=["title" , "year"])["movieId"]
recomend = recomend[~recomend.index.isin(watched_ids)]

In [28]:
user_profile

,title,year,rating
movieId,,,
6,Heat,1995,-2.0
47,Seven (a.k.a. Se7en),1995,0.0
150,Apollo 13,1995,3.5
109487,Interstellar,2014,5.0
122882,Mad Max: Fury Road,2015,2.5
146656,Creed,2015,3.0


In [58]:
licked_year = user_profile["year"].astype(float).to_numpy()
licked_weight = user_profile["rating"].astype(float).to_numpy()

years = df.set_index("movieId")["year"].astype(float).loc[recomend.index]

diff =   years.to_numpy()[ : ,None ] - licked_year[ None , :] 

sigma = 3 
kernel =  1 / (1 + np.abs(diff)/3)

score = kernel @ licked_weight
scaled_score = score / score.max()

years_score = pd.Series( scaled_score , index= years.index )


In [63]:
recomend = -.3 * years_score  +  0.7 * recomend 

## 11. Final recommendations

In [64]:
titles = df.set_index("movieId")["title"]
result = recomend.head(10).to_frame("score")
result["title"] = titles
result

,score,title
movieId,,
110730,0.065917,Transcendence
97752,0.222180,Cloud Atlas
110127,0.032550,Noah
97938,0.188812,Life of Pi
1019,0.417032,"20,000 Leagues Under the Sea"
329,0.330086,Star Trek: Generations
142863,0.004897,Crumbs
3471,0.398700,Close Encounters of the Third Kind
148080,0.370147,Hard to Be a God
